# 01 — Signal Simulation & EDA

Generate the synthetic dataset and validate that the simulator produces physiologically plausible diaphragmatic EMG signals with visible fatigue signatures.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

DATA_DIR = ROOT / 'data' / 'synthetic'
RESULTS_DIR = ROOT / 'results'

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy.signal

from simulate import simulate_window, generate_dataset, PROFILES, FS, WINDOW_S
from features import find_bursts, burst_mdf

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

## Generate dataset

1000 samples × 5 fatigue profiles (200 each). Takes ~2 minutes.

In [ ]:
X, y = generate_dataset(n_samples=1000, out_dir=DATA_DIR, seed=42)
print(f'X shape: {X.shape}   y shape: {y.shape}')
print(f'y columns: fatigue_end, mean_fatigue, mdf_slope_norm, spectral_comp, entropy')
print(f'y ranges: min={y.min(0).round(3)}  max={y.max(0).round(3)}')

## Raw signal — one sample per fatigue profile

Expect: amplitude increases then levels off; signal appears more 'rounded' (lower frequency) at higher fatigue.

In [ ]:
profile_labels = ['Fresh (0→0.05)', 'Mild (0→0.4)', 'Moderate (0.1→0.6)', 'Severe (0.3→0.8)', 'Exhausted (0.7→1.0)']
fig, axes = plt.subplots(5, 1, figsize=(14, 10), sharex=True)
t = np.arange(int(WINDOW_S * FS)) / FS

for i, (ax, label) in enumerate(zip(axes, profile_labels)):
    sig = X[i]
    ax.plot(t, sig, lw=0.4, color=plt.cm.RdYlGn_r(i / 4))
    ax.set_ylabel(label, fontsize=8)
    ax.set_ylim(-4, 4)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Synthetic diaphragmatic EMG — 30 s windows by fatigue profile', y=1.01)
plt.tight_layout()

## Spectrogram comparison — fresh vs exhausted

Expect: energy concentrated at higher frequencies (green band around 100 Hz) in the fresh signal, shifting toward lower frequencies in the exhausted signal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for ax, idx, title in zip(axes, [0, 4], ['Fresh (0→0.05)', 'Exhausted (0.7→1.0)']):
    f, t_s, Sxx = scipy.signal.spectrogram(X[idx], fs=FS, nperseg=256, noverlap=192)
    mask = (f >= 20) & (f <= 500)
    ax.pcolormesh(t_s, f[mask], 10 * np.log10(Sxx[mask] + 1e-12), shading='gouraud', cmap='inferno')
    ax.set_title(title)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Frequency (Hz)')
    ax.axhline(100, color='cyan', lw=0.8, ls='--', label='MDF fresh ~100 Hz')
    ax.axhline(45, color='lime', lw=0.8, ls='--', label='MDF exhausted ~45 Hz')
    ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Spectrogram: power spectral density (dB)')
plt.tight_layout()

## Burst detection

Validate that `find_bursts()` correctly identifies inspiratory bursts from the amplitude envelope.

In [ ]:
sig = X[2]  # moderate fatigue profile
bursts = find_bursts(sig, FS)
t = np.arange(len(sig)) / FS

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t, sig, lw=0.4, color='steelblue', label='EMG')
for s, e in bursts:
    ax.axvspan(s / FS, e / FS, alpha=0.2, color='orange')
ax.set_xlabel('Time (s)')
ax.set_title(f'Burst detection — {len(bursts)} bursts found (moderate fatigue profile)')
ax.legend()
plt.tight_layout()

## Per-burst MDF trajectory

Core signal: MDF should decrease monotonically across bursts as fatigue progresses.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
colors = plt.cm.RdYlGn_r(np.linspace(0, 1, 5))

for i, (label, color) in enumerate(zip(profile_labels, colors)):
    sig = X[i]
    bursts = find_bursts(sig, FS)
    if len(bursts) < 2:
        continue
    mdf_vals = [burst_mdf(sig[s:e], FS) for s, e in bursts]
    ax.plot(mdf_vals, marker='o', ms=3, lw=1.2, color=color, label=label)

ax.axhline(100, color='gray', lw=0.8, ls='--', label='MDF fresh baseline')
ax.axhline(45, color='gray', lw=0.8, ls=':', label='MDF exhausted baseline')
ax.set_xlabel('Burst index')
ax.set_ylabel('Median frequency (Hz)')
ax.set_title('Per-burst MDF trajectory by fatigue profile')
ax.legend(fontsize=8)
plt.tight_layout()

## Label distributions

Confirm the 5 concept bottleneck labels span their intended ranges across the dataset.

In [ ]:
label_names = ['fatigue_end', 'mean_fatigue', 'mdf_slope_norm', 'spectral_comp', 'entropy']
fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for ax, name, col in zip(axes, label_names, y.T):
    ax.hist(col, bins=30, color='steelblue', edgecolor='white', lw=0.3)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('Value')

fig.suptitle('Label distributions across 1000 samples')
plt.tight_layout()
print('Dataset ready. Proceed to 02-feature-extraction.ipynb')